# 🧠 InternHunt — Soft Computing Resume Classifier

**Upgrade: Logistic Regression → Neural Network (MLPClassifier)**

| Component | Old | New |
|-----------|-----|-----|
| Classifier | Logistic Regression (linear) | MLPClassifier — 128→64 hidden layers (non-linear) |
| TF-IDF features | 5000, max_df=0.95 | 6000, max_df=0.9, sublinear_tf=True |
| Output | Single prediction string | Top-3 predictions with fuzzy confidence labels |
| SC Concept | Hard computing / linear model | Neural Network + Fuzzy Logic |

**Fuzzy Labels:** probability > 0.7 → `High` | 0.4–0.7 → `Medium` | < 0.4 → `Low`

In [1]:
# 1) Setup
!pip -q install scikit-learn pandas joblib

In [2]:
# 2) Upload the CSV (choose UpdatedResumeDataSet.csv in the dialog)
from google.colab import files
uploaded = files.upload()

Saving UpdatedResumeDataSet.csv to UpdatedResumeDataSet.csv


In [3]:
# 3) Pin scikit-learn to a stable version for reproducibility
!pip install --upgrade scikit-learn==1.7.2
import sklearn
print("✅ Using scikit-learn version:", sklearn.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 66.6 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
✅ Using scikit-learn version: 1.7.2


In [6]:
# ============================================================
# 4) UPGRADED MODEL TRAINING  (Soft Computing Version)
#    Classifier : MLPClassifier (Neural Network, 128->64)
#    TF-IDF     : max_features=6000, sublinear_tf=True
#    Output     : Top-3 predictions + fuzzy confidence labels
# ============================================================

import pandas as pd
import re
import joblib
import sklearn

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import Pipeline

# ── Config ────────────────────────────────────────────────
CSV               = "UpdatedResumeDataSet.csv"
TEXT_COL          = "Resume"
LABEL_COL         = "Category"
MIN_SAMPLES_PER_CLASS = 4
TEST_SIZE         = 0.2
RANDOM_STATE      = 42


# ── Fuzzy Label (Soft Computing concept) ──────────────────
def fuzzy_label(probability: float) -> str:
    """
    Convert a raw probability into a fuzzy linguistic confidence label.

    Membership regions:
      High   : p > 0.70  — dominant, strongly confident prediction
      Medium : 0.40 ≤ p ≤ 0.70  — uncertain overlap zone
      Low    : p < 0.40  — weak secondary signal
    """
    if probability > 0.70:
        return "High"
    elif probability >= 0.40:
        return "Medium"
    else:
        return "Low"


# ── Text Cleaning ─────────────────────────────────────────
def clean(t: str) -> str:
    t = str(t)
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t


# ── Preprocessing ─────────────────────────────────────────
def preprocess_data(df, text_col, label_col, min_samples=MIN_SAMPLES_PER_CLASS):
    print("Original data shape:", df.shape)
    df[text_col] = df[text_col].astype(str).map(clean)
    print("After cleaning (duplicates kept):", df.shape)
    class_counts = df[label_col].value_counts()
    print(f"\nOriginal class distribution:\n{class_counts}")
    valid_classes = class_counts[class_counts >= min_samples].index
    df_filtered = df[df[label_col].isin(valid_classes)].reset_index(drop=True)
    removed = set(class_counts.index) - set(valid_classes)
    if removed:
        print(f"\nRemoved classes with < {min_samples} samples: {removed}")
    print(f"\nFinal class distribution:\n{df_filtered[label_col].value_counts()}")
    print("Final data shape:", df_filtered.shape)
    return df_filtered


# ── Pipeline (FIXED) ──────────────────────────────────────
# BUG FIX: early_stopping=True and validation_fraction removed.
#   sklearn 1.7.x bug: np.isnan() called on string class labels during
#   early-stopping validation raises TypeError.
#   max_iter raised to 500 to ensure convergence without early stopping.
def create_model_pipeline():
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=6000,      # UPGRADED: was 5000
            ngram_range=(1, 2),     # unigrams + bigrams
            min_df=2,
            max_df=0.90,            # UPGRADED: was 0.95
            sublinear_tf=True,      # UPGRADED: log(1 + tf) scaling
            stop_words='english',
            lowercase=True
        )),
        ('clf', MLPClassifier(
            hidden_layer_sizes=(128, 64),   # two hidden layers
            activation='relu',              # non-linear activation
            solver='adam',                  # adaptive moment estimation
            max_iter=500,                   # FIXED: 300→500, no early_stopping
            random_state=RANDOM_STATE
        ))
    ])


# ── Evaluation ────────────────────────────────────────────
def evaluate_model(pipeline, X_train, X_test, y_train, y_test):
    print("\n" + "="*60)
    print("TRAINING AND EVALUATION  (Neural Network — MLP 128→64)")
    print("="*60)
    print("Training... (may take 30–90 seconds)")
    pipeline.fit(X_train, y_train)

    clf_step = pipeline.named_steps['clf']
    print(f"  Completed iterations: {clf_step.n_iter_} / {clf_step.max_iter}")

    y_pred   = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nTest Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

    print("\nPerforming 5-fold cross-validation...")
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
    print(f"CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std()*2:.4f}")

    print(f"\nDetailed Classification Report:")
    print("-" * 60)
    print(classification_report(y_test, y_pred, zero_division=0))

    return pipeline, accuracy


# ── Fuzzy Prediction Output ───────────────────────────────
def predict_with_fuzzy_labels(pipeline, resume_text: str):
    """
    Returns top-3 predictions as:
        [(category, probability, fuzzy_label), ...]
    """
    probabilities = pipeline.predict_proba([resume_text])[0]
    classes       = pipeline.classes_
    top3_idx      = probabilities.argsort()[-3:][::-1]
    return [
        (classes[i], round(float(probabilities[i]), 4), fuzzy_label(probabilities[i]))
        for i in top3_idx
    ]


# ── Main ──────────────────────────────────────────────────
def main():
    print("Resume Classification — Soft Computing Upgrade (Neural Network)")
    print("=" * 65)

    try:
        df = pd.read_csv(CSV)[[TEXT_COL, LABEL_COL]].dropna()
        print(f"Loaded {len(df)} samples from {CSV}")
    except FileNotFoundError:
        print(f"Error: Could not find {CSV}"); return
    except KeyError as e:
        print(f"Error: Column not found - {e}"); return

    df_processed = preprocess_data(df, TEXT_COL, LABEL_COL)
    if len(df_processed) < 20:
        print("Error: Not enough data after preprocessing"); return

    try:
        X_train, X_test, y_train, y_test = train_test_split(
            df_processed[TEXT_COL], df_processed[LABEL_COL],
            test_size=TEST_SIZE, stratify=df_processed[LABEL_COL],
            random_state=RANDOM_STATE
        )
    except ValueError as e:
        print(f"Stratification failed: {e} — using regular split")
        X_train, X_test, y_train, y_test = train_test_split(
            df_processed[TEXT_COL], df_processed[LABEL_COL],
            test_size=TEST_SIZE, random_state=RANDOM_STATE
        )
    print(f"\nTrain set: {len(X_train)} | Test set: {len(X_test)}")

    pipeline = create_model_pipeline()
    trained_pipeline, final_accuracy = evaluate_model(
        pipeline, X_train, X_test, y_train, y_test
    )

    print(f"\n{'='*60}\nFINAL RESULTS\n{'='*60}")
    print(f"Final Test Accuracy : {final_accuracy*100:.2f}%")
    print(f"Classes             : {len(df_processed[LABEL_COL].unique())}")
    print(f"Total samples       : {len(df_processed)}")
    print(f"Model               : MLP(128, 64) + TF-IDF(6000)")

    model_filename = "resume_classifier_v2.pkl"
    metadata = {
        "model"          : trained_pipeline,
        "sklearn_version": sklearn.__version__,
        "model_type"     : "MLPClassifier",
        "architecture"   : "TF-IDF(6000) → MLP(128,64) → softmax",
        "fuzzy_labels"   : {"High": ">0.70", "Medium": "0.40-0.70", "Low": "<0.40"}
    }
    joblib.dump(metadata, model_filename)
    print(f"\n✅ Model saved to: {model_filename} (scikit-learn {sklearn.__version__})")
    return trained_pipeline


model = main()


# ── Demo: Fuzzy Output ────────────────────────────────────
sample = """
Experienced Data Scientist with 3 years in machine learning and analytics.
Proficient in Python, pandas, scikit-learn, TensorFlow, SQL, and Tableau.
Built predictive models for customer churn (AUC 0.91) and demand forecasting.
"""

print("\n" + "="*55)
print("SOFT COMPUTING PREDICTION — Fuzzy Confidence Output")
print("="*55)

results = predict_with_fuzzy_labels(model, sample)
print(f"\n{'Rank':<6} {'Category':<28} {'Probability':<14} {'Fuzzy Label'}")
print("-" * 60)
for rank, (cat, prob, label) in enumerate(results, 1):
    bar = "█" * int(prob * 20)
    print(f"  #{rank}   {cat:<28} {prob:.4f} ({prob*100:.1f}%)  [{label}]  {bar}")

print("\n🔑 Legend:")
print("  High   → p > 0.70  (dominant prediction)")
print("  Medium → 0.40–0.70 (uncertain / alternative role)")
print("  Low    → p < 0.40  (weak secondary signal)")

Resume Classification — Soft Computing Upgrade (Neural Network)
Loaded 962 samples from UpdatedResumeDataSet.csv
Original data shape: (962, 2)
After cleaning (duplicates kept): (962, 2)

Original class distribution:
Category
Java Developer               84
Testing                      70
DevOps Engineer              55
Python Developer             48
Web Designing                45
HR                           44
Hadoop                       42
Sales                        40
Data Science                 40
Mechanical Engineer          40
ETL Developer                40
Blockchain                   40
Operations Manager           40
Arts                         36
Database                     33
Health and fitness           30
PMO                          30
Electrical Engineering       30
Business Analyst             28
DotNet Developer             28
Automation Testing           26
Network Security Engineer    25
Civil Engineer               24
SAP Developer                24
Advocat

In [7]:
# ============================================================
# 5) DEMO — Fuzzy Prediction Output
#    Shows the new output format using a sample resume snippet.
# ============================================================

sample_resume = """
Experienced Data Scientist with 3 years in machine learning and analytics.
Proficient in Python, pandas, scikit-learn, TensorFlow, SQL, and Tableau.
Built predictive models for customer churn (AUC 0.91) and demand forecasting.
Strong background in statistical analysis, A/B testing, and data visualization.
"""

print("=" * 55)
print("SOFT COMPUTING PREDICTION — Fuzzy Confidence Output")
print("=" * 55)
print(f"Input: {sample_resume.strip()[:80]}...\n")

results = predict_with_fuzzy_labels(model, sample_resume)

print(f"{'Rank':<6} {'Category':<28} {'Probability':<14} {'Fuzzy Label'}")
print("-" * 60)
for rank, (category, prob, label) in enumerate(results, 1):
    bar = "█" * int(prob * 20)
    print(f"  #{rank}   {category:<28} {prob:.4f} ({prob*100:.1f}%)  [{label}]  {bar}")

print("\n🔑 Legend:")
print("  High   → probability > 0.70  (dominant prediction)")
print("  Medium → 0.40 – 0.70         (uncertain / alternative role)")
print("  Low    → < 0.40              (distant secondary signal)")

print("\n📦 Output format used in App.py:")
print("  [", results, "]")

SOFT COMPUTING PREDICTION — Fuzzy Confidence Output
Input: Experienced Data Scientist with 3 years in machine learning and analytics.
Profi...

Rank   Category                     Probability    Fuzzy Label
------------------------------------------------------------
  #1   Data Science                 0.9147 (91.5%)  [High]  ██████████████████
  #2   Arts                         0.0113 (1.1%)  [Low]  
  #3   Sales                        0.0109 (1.1%)  [Low]  

🔑 Legend:
  High   → probability > 0.70  (dominant prediction)
  Medium → 0.40 – 0.70         (uncertain / alternative role)
  Low    → < 0.40              (distant secondary signal)

📦 Output format used in App.py:
  [ [(np.str_('Data Science'), 0.9147, 'High'), (np.str_('Arts'), 0.0113, 'Low'), (np.str_('Sales'), 0.0109, 'Low')] ]


In [8]:
# ============================================================
# 6) Download the upgraded model
# ============================================================
from google.colab import files
files.download("resume_classifier_v2.pkl")
print("✅ Download started — replace the existing resume_classifier_v2.pkl in your project root.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started — replace the existing resume_classifier_v2.pkl in your project root.


# 📊 Performance Metrics

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("UpdatedResumeDataSet.csv")[["Resume", "Category"]]
X_train, X_test, y_train, y_test = train_test_split(
    df["Resume"], df["Category"], test_size=0.2, stratify=df["Category"], random_state=42
)

In [10]:
y_pred = model.predict(X_test)

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_test, y_pred, average='weighted', zero_division=0)
f1        = f1_score(y_test, y_pred, average='weighted', zero_division=0)

print("Model Type : Neural Network (MLPClassifier, 128→64 hidden layers)")
print("-" * 55)
print("Accuracy  :", round(accuracy, 3))
print("Precision :", round(precision, 3))
print("Recall    :", round(recall, 3))
print("F1 Score  :", round(f1, 3))

Model Type : Neural Network (MLPClassifier, 128→64 hidden layers)
-------------------------------------------------------
Accuracy  : 1.0
Precision : 1.0
Recall    : 1.0
F1 Score  : 1.0


In [12]:
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Classification Report:
                            precision    recall  f1-score   support

                 Advocate       1.00      1.00      1.00         4
                     Arts       1.00      1.00      1.00         7
       Automation Testing       1.00      1.00      1.00         5
               Blockchain       1.00      1.00      1.00         8
         Business Analyst       1.00      1.00      1.00         6
           Civil Engineer       1.00      1.00      1.00         5
             Data Science       1.00      1.00      1.00         8
                 Database       1.00      1.00      1.00         7
          DevOps Engineer       1.00      1.00      1.00        11
         DotNet Developer       1.00      1.00      1.00         5
            ETL Developer       1.00      1.00      1.00         8
   Electrical Engineering       1.00      1.00      1.00         6
                       HR       1.00      1.00      1.00         9
                   Hadoop       1.00